# Stochastic Depth — Deep Networks with Stochastic Depth (2016)

_Papers · Computer Vision_

**Randomly delete whole residual blocks during training (keep only the skip); train shorter, test full.**

---

This notebook is a practice scaffold for the **Stochastic Depth — Deep Networks with Stochastic Depth (2016)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. These are **images, not table columns**. We pull a few raw samples from the CIFAR10 dataset to see them before any transforms.

In [ ]:
import torchvision

preview = torchvision.datasets.CIFAR10(root="./data", train=True, download=True)
print("dataset: CIFAR10   samples:", len(preview))
first_image, first_label = preview[0]
print("one sample:", first_image.size, "image,  label =", first_label)
print("classes:", getattr(preview, "classes", "(digit labels 0-9)"))
samples = [preview[i] for i in range(5)]
fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for ax, (image, label) in zip(axes, samples):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.show()

## Reference implementation — PyTorch

In [ ]:
import torch
import torch.nn as nn
import torchvision, torchvision.transforms as T
import time

torch.manual_seed(0)

# --- 0. Sanity-check the worked example. -------------------------------------
L, pL = 5, 0.5
p = [1 - (l / L) * (1 - pL) for l in range(1, L + 1)]   # Eqn. 4 schedule
print("survival schedule p_1..p_5:", [round(v, 3) for v in p])      # [0.9,0.8,0.7,0.6,0.5]
print("E[depth] = sum p_l        :", round(sum(p), 3),
      " | (3L-1)/4 =", (3 * L - 1) / 4)                              # 3.5 | 3.5
H1, f2, p2 = 2.0, 0.6, p[1]                                          # block l=2, p_2=0.8
print("train survive relu(f+H)   :", max(0.0, 1 * f2 + H1))         # 2.6
print("train drop    relu(H)     :", max(0.0, 0 * f2 + H1))         # 2.0
print("train average H + p2*f    :", round(H1 + p2 * f2, 3))        # 2.48
print("test  relu(p2*f + H)      :", round(max(0.0, p2 * f2 + H1), 3))  # 2.48 (== train avg)


# --- 1. The stochastic-depth residual block (built by hand). -----------------
class StochasticBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1, p_survive=1.0):
        super().__init__()
        self.p = p_survive                       # survival probability for THIS block
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, stride=1, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_ch)
        self.relu  = nn.ReLU(inplace=True)
        self.proj = None                         # projection skip when shape changes
        if stride != 1 or in_ch != out_ch:
            self.proj = nn.Sequential(nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                                      nn.BatchNorm2d(out_ch))

    def f(self, x):                              # the residual function f_l
        out = self.relu(self.bn1(self.conv1(x)))
        return self.bn2(self.conv2(out))

    def forward(self, x):
        identity = self.proj(x) if self.proj is not None else x
        if self.training:
            # Bernoulli coin per block (shared across the mini-batch).
            if torch.rand(1).item() < self.p:    # b_l = 1  -> Eqn. 2
                return self.relu(self.f(x) + identity)
            else:                                # b_l = 0  -> Eqn. 3: skip f entirely
                return self.relu(identity)       # NOTE: convolutions are NOT run -> faster
        else:
            # Test time (Eqn. 5): keep the block, scale the residual by p.
            return self.relu(self.p * self.f(x) + identity)


# --- 2. Stack blocks; assign the linear survival schedule by global depth. ---
class SmallResNet(nn.Module):
    def __init__(self, blocks_per_stage=3, pL=0.5, n_classes=10):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(3, 16, 3, padding=1, bias=False),
                                  nn.BatchNorm2d(16), nn.ReLU(inplace=True))
        widths = [(16, 16, 1), (16, 32, 2), (32, 64, 2)]
        Ltot = len(widths) * blocks_per_stage           # total residual blocks
        blocks, idx = [], 0
        for (ci, co, st) in widths:
            for b in range(blocks_per_stage):
                idx += 1
                ps = 1.0 if pL >= 1.0 else 1 - (idx / Ltot) * (1 - pL)   # Eqn. 4
                blocks.append(StochasticBlock(ci if b == 0 else co, co,
                                              st if b == 0 else 1, ps))
        self.blocks = nn.Sequential(*blocks)
        self.head   = nn.Linear(64, n_classes)

    def forward(self, x):
        x = self.blocks(self.stem(x))
        return self.head(x.mean(dim=(2, 3)))             # global average pool


# --- 3. A CIFAR-10 subset (torchvision is preinstalled in Colab). ------------
tfm = T.Compose([T.ToTensor(),
                 T.Normalize((0.4914, 0.4822, 0.4465), (0.247, 0.243, 0.261))])
full = torchvision.datasets.CIFAR10(root="./data", train=True, download=True, transform=tfm)
train_set = torch.utils.data.Subset(full, range(4000))
test_set  = torch.utils.data.Subset(full, range(4000, 5000))
tr = torch.utils.data.DataLoader(train_set, batch_size=128, shuffle=True)
te = torch.utils.data.DataLoader(test_set,  batch_size=256)
device = "cuda" if torch.cuda.is_available() else "cpu"


def run(pL, epochs=4, depth=3):
    torch.manual_seed(0)
    net = SmallResNet(blocks_per_stage=depth, pL=pL).to(device)
    opt = torch.optim.SGD(net.parameters(), lr=0.05, momentum=0.9, weight_decay=5e-4)
    lf  = nn.CrossEntropyLoss()
    t0 = time.time()
    for ep in range(epochs):
        net.train()
        for xb, yb in tr:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad(); lf(net(xb), yb).backward(); opt.step()
    secs = time.time() - t0
    net.eval(); correct = tot = 0
    with torch.no_grad():
        for xb, yb in te:
            pred = net(xb.to(device)).argmax(1).cpu()
            correct += (pred == yb).sum().item(); tot += yb.numel()
    return secs, 100 * (1 - correct / tot)               # wall-clock, test error %

print("\nSTOCHASTIC depth (p_L=0.5):")
s_secs, s_err = run(pL=0.5)
print("CONSTANT depth   (p_L=1.0, ABLATION  -- no dropping):")
c_secs, c_err = run(pL=1.0)
print(f"\n  stochastic: {s_secs:5.1f}s/4ep  test err {s_err:5.2f}%")
print(f"  constant  : {c_secs:5.1f}s/4ep  test err {c_err:5.2f}%")
# Stochastic depth trains faster (fewer blocks run per step) and tends to generalize
# a bit better. Exact numbers vary by hardware/seed -- our small run, not the paper's.

## Visualize the data & results

_Does randomly dropping residual blocks during training speed it up AND lower test error vs constant depth?_

In [ ]:
import torch, torch.nn as nn, numpy as np

# Toy reproduction: does stochastic depth train faster + regularize vs constant depth?
N, K, n = 400, 4, 24
g = torch.Generator().manual_seed(2)
y = torch.randint(0, K, (N,), generator=g)
centers = torch.randn(K, n, generator=g) * 1.5
X = centers[y] + torch.randn(N, n, generator=g) * 0.8
Xtr, ytr, Xte, yte = X[:300], y[:300], X[300:], y[300:]

class Block(nn.Module):
    def __init__(self, n, p):
        super().__init__()
        self.fc = nn.Linear(n, n, bias=False); self.bn = nn.BatchNorm1d(n); self.p = p
    def f(self, x):  return self.bn(self.fc(x))
    def forward(self, x):
        if self.training:
            if torch.rand(1).item() < self.p:   return torch.relu(self.f(x) + x)  # Eqn. 2
            return torch.relu(x)                                                  # Eqn. 3: drop f
        return torch.relu(self.p * self.f(x) + x)                                 # Eqn. 5: scale

class Net(nn.Module):
    def __init__(self, pL, Lblocks=9):
        super().__init__()
        self.stem = nn.Linear(n, n)
        ps = [1.0 if pL >= 1 else 1 - (l / Lblocks) * (1 - pL) for l in range(1, Lblocks + 1)]
        self.blocks = nn.Sequential(*[Block(n, p) for p in ps])
        self.head = nn.Linear(n, K); self.Edepth = sum(ps)
    def forward(self, x): return self.head(self.blocks(torch.relu(self.stem(x))))

def run(pL, epochs=8):
    torch.manual_seed(0)
    net = Net(pL); opt = torch.optim.SGD(net.parameters(), lr=0.05, momentum=0.9)
    lf = nn.CrossEntropyLoss(); test_losses = []
    for ep in range(epochs):
        net.train(); opt.zero_grad(); lf(net(Xtr), ytr).backward(); opt.step()
        net.eval()
        with torch.no_grad(): test_losses.append(round(lf(net(Xte), yte).item(), 4))
    return test_losses, net.Edepth

const_loss, _      = run(1.0)
stoch_loss, Edepth = run(0.5)
print("Constant   test loss/epoch:", const_loss)
print("Stochastic test loss/epoch:", stoch_loss)
print("Expected active blocks E(L~):", round(Edepth, 2), "of 9  -> ~%.2fx forward cost" % (Edepth / 9))
# Stochastic depth descends faster to a lower test loss and runs ~0.72x the
# blocks per step (speedup). Our small run, not the paper's reported numbers.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The ablation. You have a small ResNet that trains with the stochastic-depth schedule on
            ($p_L = 0.5$). Turn the schedule off by setting every $p_\ell = 1$ (ordinary constant-depth
            ResNet), keeping depth, width, optimizer, data, and seed identical. What changes in (a) wall-clock
            per epoch and (b) test error, and what does each change demonstrate?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Set every block's survival probability to $1$ so no block is ever dropped; change nothing else. — _An honest ablation flips exactly one knob &mdash; the random dropping &mdash; so any difference is attributable to stochastic depth._
- Time an epoch each way. With the schedule on, about $E(\tilde{L})=\sum p_\ell \approx 3L/4$ blocks run per step, so fewer convolutions execute &rarr; faster. — _Dropped blocks do no forward/backward work; the expected active depth is ~three-quarters of $L$, which is the source of the wall-clock saving._
- Compare final test error. The stochastic-depth run typically generalizes a little better. — _Random dropping is a regularizer (an implicit ensemble over many sub-depths), much like Dropout &mdash; see paper-dropout._

**Answer:** (a) Per-epoch wall-clock drops with the schedule on, because on average only
                 $\sum_\ell p_\ell \approx 3L/4$ blocks execute per step &mdash; the dropped blocks skip their
                 convolutions. (b) Test error is usually a bit lower with stochastic depth, because the
                 random dropping regularizes (an implicit ensemble of shorter networks). Same net otherwise, so
                 both effects are attributable to the dropping. The paper's Table 1 shows the same direction
                 (e.g. CIFAR-10 6.41% &rarr; 5.25%); our CODEVIZ panel reproduces the qualitative effect on a
                 toy run.

</details>

**Problem 2.** For a network with $L = 10$ residual blocks and $p_L = 0.5$, compute the survival probability of
            block $\ell = 4$ and the expected number of active blocks. Then give block 4's test-time
            output if its input is $H_3 = 1.0$ and its residual is $f_4(H_3) = 0.5$.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Eqn. 4: $p_4 = 1 - (4/10)(1 - 0.5) = 1 - 0.4(0.5) = 1 - 0.2 = 0.8$. — _Plug $\ell = 4$, $L = 10$, $p_L = 0.5$ into the linear schedule._
- Expected depth $E(\tilde{L}) = \sum_{\ell=1}^{10} \big(1 - 0.05\,\ell\big)$. The closed form $(3L-1)/4 = (30-1)/4 = 7.25$. — _Sum of the linear schedule; equivalently the $(3L-1)/4$ formula from &sect;3._
- Test forward (Eqn. 5): $H_4^{\text{Test}} = \mathrm{ReLU}(p_4\, f_4 + H_3) = \mathrm{ReLU}(0.8(0.5) + 1.0) = \mathrm{ReLU}(1.4) = 1.4$. — _At test all blocks are on but the residual is scaled by its survival probability $p_4 = 0.8$._

**Answer:** $p_4 = 0.8$; expected active blocks $E(\tilde{L}) = (3\cdot 10 - 1)/4 = 7.25$ of $10$; and
                 block 4's test output is $\mathrm{ReLU}(0.8\cdot 0.5 + 1.0) = \mathbf{1.4}$. The training
                 average would be $1.0 + 0.8(0.5) = 1.4$ as well &mdash; the $\times p_\ell$ scale makes test
                 equal the training expectation.

</details>

**Problem 3.** A teammate "saves the rescale for later" and ships a model that, at test time, keeps every block
            on but uses the raw residual $f_\ell$ (no $\times p_\ell$). Validation accuracy is much worse
            than during training. What went wrong and what is the one-line fix?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Compare train vs test residual magnitude. In training block $\ell$'s residual averaged $p_\ell\,f_\ell$; at test, unscaled, it is $f_\ell$ &mdash; a factor $1/p_\ell$ too large. — _The downstream layers were tuned to the smaller, $p_\ell$-scaled average, so unscaled residuals push activations off-distribution._
- These over-large residuals compound across blocks, shifting the feature statistics the BatchNorm / classifier expect. — _Each block adds an inflated residual; errors accumulate with depth._
- Apply Eqn. 5: in eval mode scale every residual by its survival probability &mdash; relu(p * f(x) + identity). — _That makes the deterministic test pass equal the training expectation, exactly as Dropout rescales at inference._

**Answer:** Without the $\times p_\ell$ scale, every residual at test time is about $1/p_\ell$ too large
                 relative to what training produced on average, so activations drift off-distribution and
                 accuracy collapses. The fix is Eqn. 5: in eval mode use relu(p * f(x) + identity).
                 It is the same expectation-matching rescale as Dropout (paper-dropout).

</details>